# <center>Singular Value Decomposition: Theory, Implementation and a MovieLens Case Study

## Abstract

## Contents

## Introduction
The singular value decomposition (SVD) is a powerful matrix factorization technique in linear algebra, providing a foundation for many data methods such as Principal Component Analysis (PCA), the Dynamic Mode Decomposition (DMD), and the Proper Orthogonal Decomposition (POD). The SVD provides a numerically stable matrix decomposition that can be used for a variety of purposes and is guaranteed to exist for every matrix.


## SVD Mathematical Overview

Let $X \in \mathbb{R}^{n \times m}$ be a data matrix with $n$ rows and $m$ columns.

**Note:** The SVD also exists for complex matrices $X \in \mathbb{C}^{n \times m}$, with the transpose replaced by the conjugate transpose. For simplicity, all examples and proofs in this work use real matrices.

$$X = \begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1m} \\
x_{21} & x_{22} & \cdots & x_{2m} \\
\vdots & \vdots & \ddots & \vdots \\
x_{n1} & x_{n2} & \cdots & x_{nm}
\end{bmatrix}$$

The columns of $X$ are typically measurements collected from experiments or simulations. For instance, each column can be an image that has been reshaped into a long vector with one entry per pixel. A column can also describe the state of a physical system at one moment in time, such as the fluid velocity at a set of grid points, a set of neural recordings, or the state of a weather simulation at a resolution of one square kilometre. Putting many such snapshots side by side gives the matrix $X$: its $m$ columns are the snapshots, and its $n$ rows are the entries of each snapshot.

Data of this kind is high-dimensional: a single megapixel image has a million entries, and a high-resolution video contains thousands of such images. The SVD is a general-purpose tool of numerical linear algebra for processing such data. It reduces a large data set to a small number of key features that are sufficient to analyse, understand and describe it. For this reason it is often the first step in dimensionality-reduction and machine-learning pipelines

The SVD can also be seen as a data-driven generalisation of the Fourier transform. Both map a system into a new coordinate system in which its behaviour becomes simpler. The Fourier transform uses a fixed, predefined set of waves. For many modern systems, for example the turbulent flow of air over an aircraft wing, there is no ready-made transformation that describes the behaviour well. The SVD solves this by building the coordinate system from the data itself, so the transformation is tailored to a specific problem. In this sense the SVD is a data-reduction algorithm: it compresses large data sets into their most informative components.

Depending on the application, the data matrix $X$ can have very different shapes. If it has many more rows than columns ($n \gg m$), it is called *tall and skinny*. If it has many more columns than rows ($n \ll m$), it is called *short and fat*.

**Note:** In this work we consider the tall and skinny case, $m \ll n$. For example, $n$ is the number of pixels in an image and $m$ is the number of images.

The SVD exists for every matrix, whatever its shape, and it is unique up to the sign and ordering conventions. It decomposes $X$ as

$$X = U \Sigma V^\top = \begin{bmatrix} \vert & \vert & & \vert \\ u_1 & u_2 & \cdots & u_n \\ \vert & \vert & & \vert \end{bmatrix} \left[ \begin{array}{c} \begin{array}{cccc} \sigma_1 & & & \\ & \sigma_2 & & \\ & & \ddots & \\ & & & \sigma_m \end{array} \\ \hline \Large 0 \end{array} \right] \begin{bmatrix} \vert & \vert & & \vert \\ v_1 & v_2 & \cdots & v_m \\ \vert & \vert & & \vert \end{bmatrix}^\top = \begin{bmatrix} \vert & \vert & & \vert \\ u_1 & u_2 & \cdots & u_n \\ \vert & \vert & & \vert \end{bmatrix} \left[ \begin{array}{c} \begin{array}{cccc} \sigma_1 & & & \\ & \sigma_2 & & \\ & & \ddots & \\ & & & \sigma_m \end{array} \\ \hline \Large 0 \end{array} \right] \begin{bmatrix} \text{—} & v_1^\top & \text{—} \\ \text{—} & v_2^\top & \text{—} \\ & \vdots & \\ \text{—} & v_m^\top & \text{—} \end{bmatrix} \tag{1}$$

where

- $U \in \mathbb{R}^{n \times n}$ and $V \in \mathbb{R}^{m \times m}$ are **orthogonal matrices**, which means that their columns are orthonormal;
- $\Sigma \in \mathbb{R}^{n \times m}$ is a matrix with real, non-negative entries on the diagonal and zeros everywhere else.

The following figure illustrates the matrix equation above and it is known as **Full SVD**:

![Full SVD of a tall matrix X](images/svd_full.png)

*Figure 1. Full SVD: $U$ ans $V$ are square matrices, and $\Sigma$ has the diagonal block of singular values above a block of zeros.*

So essentially $X$ is a product of three matrices where $U$ contains information of column space of $X$, $V$ contains information of the row space of $X$ and $\Sigma$ is a diagonal matrix of singular values that tells how important the various columns of $U$ and $V$ are. All theree matrices are hierachically arranged so that $u_1$ is more important than $u_2$, which is more importan than $u_3$ and so on. The same sorting is valid for $V$ and $\Sigma$ singular values. 

It is valuable to mention the SVD is guaranteed to exist for every matrix, whatever its shape. And it is unique up to flipping the sign (plus/minus) of the singular vectors.

**Orthogonal matrices:** $U$ and $V$ are orthogonal when $U^\top U = I$ and $V^\top V = I$. This means that multiplying by $U$ or $V$ only rotates or reflects vectors without changing their lengths.

The columns of $U$ are called the **left singular vectors** of $X$, and the columns of $V$ are called the **right singular vectors**. The diagonal entries $\sigma_1, \dots, \sigma_m$ of $\Sigma$ are called the **singular values** of $X$. They are non-negative and ordered from largest to smallest:

$$\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_m \ge 0 .$$

The rank of $X$ equals the number of non-zero singular values. For real-world data, the singular values measure how much of the data is captured by each component, so the components are ordered from the most to the least important.

### Economy SVD 
For the $X \in \mathbb{R}^{n \times m}$, where ($m \le n$), $\Sigma$ has at most $m$ non-zero diagonal entries, so everything below its first $m$ rows is zero:

$$\Sigma = \begin{bmatrix} \hat{\Sigma} \\ 0 \end{bmatrix}, \hat{\Sigma} \in \mathbb{R}^{m \times m}.$$

Split $U$ in the same way into $U = [\hat{U} \;\; \hat{U}^\perp]$, where $\hat{U} \in \mathbb{R}^{n \times m}$ contains the first $m$ columns and $\hat{U}^\perp \in \mathbb{R}^{n \times (n-m)}$ contains the remaining $n - m$ columns. The columns of $\hat{U}^\perp$,  are multiplied only by zeros, so they do not contribute, and $X$ can be represented exactly by the smaller **Economy SVD**:

$$X = U \Sigma V^\top = [\hat{U} \;\; \hat{U}^\perp] \begin{bmatrix} \hat{\Sigma} \\ 0 \end{bmatrix} V^\top = \hat{U} \hat{\Sigma} V^\top .$$

The following image depicts the Economy SVD:
![Economy SVD of a tall matrix X](images/svd_economy.png)

*Figure 2. Economy SVD: the zero block of $\Sigma$ and the matching columns of $U$ are dropped.*

### Matrix Approximation


The SVD is especially valuable because it provides an optimal low-rank approximation of a matrix $X$ for a specific rank $r$ where $r \lt \operatorname{rank}(X) \le \min(n, m)$. In fact, for a fixed $r$ there are infinitely many matrices of rank $r$ that approximate $X$. A rank-r approximation is obtained by keeping the leading r singular values and vectors, and discarding the rest. This is called the **truncated SVD**. The result that the truncated SVD is the optimal low-rank approximation was first established by Erhard Schmidt, who is also known for the Gram–Schmidt process and who worked with function spaces. It was later rediscovered and formulated by Eckart and Young, and it is now usually called the **Eckart–Young theorem**.

**Theorem (Eckart–Young):** Let $X \in \mathbb{R}^{n \times m}$ and $X = U \Sigma V^\top$

and let $r \le \operatorname{rank}(X)$. The optimal rank-$r$ approximation of $X$, in the least-squares sense, is the truncated SVD:

$$\underset{\tilde{X} \,:\, \operatorname{rank}(\tilde{X}) = r}{\operatorname{argmin}} \; \lVert X - \tilde{X} \rVert_F = \tilde{U} \tilde{\Sigma} \tilde{V}^\top .$$

Here $\tilde{U}$ and $\tilde{V}$ contain the first $r$ columns of $U$ and $V$, $\tilde{\Sigma}$ is the leading $r \times r$ block of $\Sigma$, and $\lVert \cdot \rVert_F$ is the Frobenius norm, $\lVert A \rVert_F = \sqrt{\sum_{i,j} a_{ij}^2}$.

We denote the truncated SVD basis, and the approximation of $X$ that results from it, by

$$\tilde{X} = \tilde{U} \tilde{\Sigma} \tilde{V}^\top .$$

Because $\Sigma$ is diagonal, the rank-$r$ approximation is the sum of $r$ distinct rank-one matrices:

$$\tilde{X} = \sum_{k=1}^{r} \sigma_k u_k v_k^\top = \sigma_1 u_1 v_1^\top + \sigma_2 u_2 v_2^\top + \dots + \sigma_r u_r v_r^\top .$$

This is called the **dyadic summation**. For a given rank $r$, no other rank-$r$ matrix approximates $X$ better, in the least-squares sense, than the truncated SVD $\tilde{X}$. As a result, high-dimensional data can often be described well by a few dominant patterns, given by the leading columns of $\tilde{U}$ and $\tilde{V}$.

The following figure illuststrates the Truncated SVD:
![Truncated SVD of a tall matrix X](images/svd_truncated.png)

*Figure 3. Truncated SVD: only the first $r$ columns and singular values are kept, so $\tilde{X}$ approximates $X$.*

## Example

## Conclusion and next steps

## References